# LOKLOK v8 — Original Data + LLM Embeddings + T4 GPU

This notebook rebuilds the final results from the raw data. It does **not** reuse any previous result file.

1. Download the private v8 bundle from the temporary tunnel.
2. Transcribe the researcher self-report with Whisper `large-v3` on the T4.
3. Embed PO, FGD, self-report, corpora and stimulus briefs with `BAAI/bge-m3`.
4. Run DML/IPW, spectral manifold, 2^3 tensor ABM, GRM-IRT, Bayesian, adversarial, Sobol and robustness.
5. Generate figures, zip outputs and download.


In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


In [ ]:
%%capture
!pip -q install -U "sentence-transformers>=3.0" "faster-whisper>=1.0" "transformers>=4.40" accelerate librosa soundfile


In [ ]:
import os, pathlib, subprocess
BUNDLE_PATH = "/content/colab_bundle_v8.zip"
if not os.path.exists(BUNDLE_PATH):
    from google.colab import files
    print("Upload colab_bundle_v8.zip from the package root")
    files.upload()
if not os.path.exists(BUNDLE_PATH):
    raise FileNotFoundError("colab_bundle_v8.zip not found in /content")
subprocess.run(["unzip", "-q", "-o", BUNDLE_PATH, "-d", "/content"], check=True)
ROOT = pathlib.Path("/content/loklok_v8")
print(ROOT, ROOT.exists())
print(sorted(p.name for p in ROOT.iterdir()))


In [ ]:
from faster_whisper import WhisperModel
from pathlib import Path
import json, time, numpy as np, librosa

ROOT = Path("/content/loklok_v8")
audio = ROOT / "03_data" / "raw" / "selfreport_16k_48k.opus"
out_dir = ROOT / "03_data" / "qualitative" / "selfreport"
out_dir.mkdir(parents=True, exist_ok=True)
print("audio", audio, audio.stat().st_size)

y, sr = librosa.load(str(audio), sr=16000, mono=True)
print("decoded", y.shape, sr, "sec", round(len(y)/sr, 1))

model = WhisperModel("large-v3", device="cuda", compute_type="int8_float16")
t0 = time.time()
segments, info = model.transcribe(
    y,
    language="ar",
    beam_size=5,
    vad_filter=True,
    condition_on_previous_text=False,
)
rows = []
for s in segments:
    txt = s.text.strip()
    if txt:
        rows.append({"start": float(s.start), "end": float(s.end), "text": txt})
print("language", info.language, "prob", info.language_probability, "duration", info.duration)
print("segments", len(rows), "words", sum(len(r["text"].split()) for r in rows), "elapsed", round(time.time()-t0,1))
(out_dir / "selfreport_transcript.json").write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
(out_dir / "selfreport_transcript.txt").write_text(chr(10).join(r["text"] for r in rows), encoding="utf-8")
print(rows[:3])


In [ ]:
import gc, torch
for name in ["asr", "model", "y", "result", "segments", "info"]:
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f"GPU free {free/1e9:.2f} GB of {total/1e9:.2f} GB; torch allocated {torch.cuda.memory_allocated()/1e9:.2f} GB")


In [ ]:
!sed -i 's/batch_size: int = 64/batch_size: int = 16/' /content/loklok_v8/02_code/pipeline/loklok_pipeline_v8.py
!grep -n "batch_size: int" /content/loklok_v8/02_code/pipeline/loklok_pipeline_v8.py


In [ ]:
!cd /content/loklok_v8 && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python 02_code/pipeline/loklok_pipeline_v8.py --root /content/loklok_v8 --n-personas 500 --sessions 18 --seed 20261105 --embedding-model BAAI/bge-m3 --device cuda


In [ ]:
!cd /content/loklok_v8 && python 02_code/pipeline/generate_figures.py --root /content/loklok_v8


In [ ]:
import json
from pathlib import Path
ROOT = Path("/content/loklok_v8")
r = json.loads((ROOT/"03_data/model_outputs/pipeline_results.json").read_text(encoding="utf-8"))
print("metadata:", json.dumps(r["metadata"], ensure_ascii=False, indent=2)[:2000])
print("\nfactorial delta means:")
for k in ["delta_sr_eval","delta_sr_comp","delta_engagement"]:
    print(k, r["factorial"][k]["mean"], r["factorial"][k].get("hdi_low"), r["factorial"][k].get("hdi_high"))
print("\ncalibration:", r["calibration"])
print("\nembedding:", {k:r["embedding"][k] for k in ["n_features_biased","n_quant_features","n_text_features","condition_after","llm_embedding_dim","semantic_components","documents"]})


In [ ]:
!cd /content && rm -f loklok_v8_results.zip && zip -q -r loklok_v8_results.zip loklok_v8/03_data/model_outputs loklok_v8/03_data/measured loklok_v8/03_data/qualitative/selfreport loklok_v8/01_thesis/src/generated loklok_v8/01_thesis/src/figures loklok_v8/04_figures
!ls -lh /content/loklok_v8_results.zip


In [ ]:
from google.colab import files
files.download("/content/loklok_v8_results.zip")
